# HabitatVLM — Backend B: Ollama on a Kaggle GPU

Runs a vision-language model (default `qwen3-vl:8b-instruct`) with **Ollama** on Kaggle's free GPU and exposes it
to your laptop through a **Cloudflare quick tunnel**. The laptop bridge talks to it with the
OpenAI-compatible API (`<tunnel>/v1`).

**Before running**
1. Right panel → *Session options* → **Accelerator: GPU T4 x2** (or P100).
2. **Internet: On** (needs a phone-verified Kaggle account).
3. *Run All*. Keep this browser tab open while you work.

**After it prints the URL**
- Put it in `bridge/.env` as `KAGGLE_TUNNEL_URL=https://....trycloudflare.com`
- Set `active_backend: kaggle` in `bridge/config.yaml` and (re)start the bridge.
- Test from the laptop: `..\my_env\Scripts\python.exe tools\fake_unity.py`

**Security:** the tunnel URL is public and unauthenticated while the session runs. Don't share it,
and **stop the session when you're done** (Run → *Stop session*). It also saves GPU quota (~30 h/week).

In [ ]:
# ---- Settings ----
MODEL = "qwen3-vl:8b-instruct"   # instruct = no hidden thinking (fast). Alternatives: "qwen3-vl:4b-instruct", "gemma3:12b"
OLLAMA_PORT = 11434

In [ ]:
# 1) Install Ollama (the installer needs zstd) and confirm the GPU is visible
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Start the Ollama server in the background
import os, subprocess, time, requests

env = dict(os.environ,
           OLLAMA_HOST=f"0.0.0.0:{OLLAMA_PORT}",
           OLLAMA_KEEP_ALIVE="-1",        # keep the model loaded in GPU memory
           OLLAMA_NUM_PARALLEL="1",
           OLLAMA_CONTEXT_LENGTH="8192",  # prompts are ~1.4k tokens; smaller context = less GPU memory
           OLLAMA_ORIGINS="*")
ollama_log = open("/kaggle/working/ollama.log", "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        v = requests.get(f"http://127.0.0.1:{OLLAMA_PORT}/api/version", timeout=2).json()
        print("Ollama is up:", v); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")

In [ ]:
# 3) Download the model (several GB the first time; a few minutes)
!ollama pull {MODEL}
!ollama list

In [ ]:
# 4) Local test with an image: also loads the model into GPU memory (warm-up)
import base64, io, json
from PIL import Image, ImageDraw

img = Image.new("RGB", (512, 512), (215, 220, 228))
d = ImageDraw.Draw(img)
d.rectangle([0, 360, 512, 512], fill=(64, 69, 77))
d.rectangle([180, 200, 260, 360], fill=(217, 26, 26))     # a red "canister"
d.rectangle([330, 160, 450, 360], fill=(128, 51, 179))    # a purple "cabinet"
buf = io.BytesIO(); img.save(buf, format="JPEG", quality=80)
b64 = base64.b64encode(buf.getvalue()).decode()

payload = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "Reply with JSON only: {\"objects\": [list of colours of the objects you see]}"},
        {"role": "user", "content": [
            {"type": "text", "text": "What coloured objects are in this image?"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}]},
    ],
    "response_format": {"type": "json_object"},
    "temperature": 0.2, "max_tokens": 200,
}
for i in range(2):   # 1st call includes model load time, 2nd is the real latency
    t0 = time.time()
    r = requests.post(f"http://127.0.0.1:{OLLAMA_PORT}/v1/chat/completions", json=payload, timeout=600)
    r.raise_for_status()
    print(f"call {i+1}: {time.time()-t0:.1f}s ->", r.json()["choices"][0]["message"]["content"])
!ollama ps

In [ ]:
# 5) Open a Cloudflare quick tunnel (no account needed) and print the public URL
import re, threading

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate",
     "--url", f"http://127.0.0.1:{OLLAMA_PORT}",
     "--http-host-header", f"localhost:{OLLAMA_PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

TUNNEL_URL = None
deadline = time.time() + 60
while time.time() < deadline and TUNNEL_URL is None:
    line = tunnel_proc.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        TUNNEL_URL = m.group(0)
if not TUNNEL_URL:
    raise RuntimeError("No tunnel URL from cloudflared - re-run this cell")

# keep draining cloudflared output so its pipe never fills up
threading.Thread(target=lambda: [None for _ in tunnel_proc.stdout], daemon=True).start()

print("=" * 70)
print("Paste this line into bridge/.env on your laptop:\n")
print(f"KAGGLE_TUNNEL_URL={TUNNEL_URL}")
print("=" * 70)

In [ ]:
# 6) Check the tunnel end-to-end (DNS for a new tunnel can take ~10-30 s)
for i in range(12):
    try:
        r = requests.get(f"{TUNNEL_URL}/v1/models", timeout=10)
        if r.ok:
            print("Tunnel OK. Models:", [m["id"] for m in r.json()["data"]]); break
    except Exception as e:
        err = e
    time.sleep(5)
else:
    print("Tunnel not reachable yet:", err, "- wait a bit and re-run this cell")

In [ ]:
# 7) Monitor: shows requests arriving from your laptop. Interrupt (stop button) to end monitoring;
#    the server and tunnel keep running until you stop the session.
last = 0
try:
    while True:
        with open("/kaggle/working/ollama.log") as f:
            f.seek(last); new = f.read(); last = f.tell()
        for line in new.splitlines():
            if "/v1/chat/completions" in line or "error" in line.lower():
                print(line[-160:])
        if ollama_proc.poll() is not None:
            print("Ollama exited!"); break
        if tunnel_proc.poll() is not None:
            print("Tunnel exited! Re-run step 5 and update KAGGLE_TUNNEL_URL."); break
        time.sleep(10)
except KeyboardInterrupt:
    print("Monitoring stopped (server and tunnel still running).")

### When you are finished
Run → **Stop session**. This shuts down Ollama and the tunnel (the URL stops working) and saves GPU quota.
Next session: *Run All* again. The tunnel URL **changes every time**, so update `bridge/.env`.